# Pre-Training on SlimPajama-6B (Azure A100)

**Model**: 154M Decoder-only Transformer (GQA + REPO-Attention + Flash-Attention)  
**Dataset**: SlimPajama-6B via Oxen  
**Tracking**: Weights & Biases

In [ ]:
# Uncomment on Azure if needed
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip install oxen wandb transformers tokenizers pyarrow

In [ ]:
import os, sys, time, math
import torch
import torch.nn as nn
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader
from datetime import datetime
import wandb

PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
TRAIN_DIR = os.path.join(PROJECT_ROOT, "train")
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, TRAIN_DIR)

from transformer.build_transformer import build_transformer
from dataset_define import SlimPajamaDataset
from save_checkpoint import save_checkpoint
from tokenizer import tokenizer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ======================== CONFIG ========================
DATASET_DIR    = os.path.join(PROJECT_ROOT, "SlimPajama-6B")
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, "checkpoints")

# Model
D_MODEL     = 768
NUM_LAYERS  = 12
NUM_HEADS   = 12
KV_HEADS    = 4
D_FF        = 3072
DROPOUT     = 0.1
MAX_SEQ_LEN = 2048
USE_REPO    = True
USE_FLASH   = True

# Training
EPOCHS         = 1
BATCH_SIZE     = 24
GRAD_ACCUM     = 8
LEARNING_RATE  = 3e-4
MIN_LR         = 3e-5     # 10% of peak
WEIGHT_DECAY   = 0.01
MAX_GRAD_NORM  = 1.0
WARMUP_STEPS   = 200      # Short warmup so model starts learning fast

# Estimated steps (for cosine schedule)
TOKENS_IN_DATASET = 6_000_000_000
TOKENS_PER_STEP   = BATCH_SIZE * GRAD_ACCUM * MAX_SEQ_LEN
EST_STEPS_PER_EPOCH = TOKENS_IN_DATASET // TOKENS_PER_STEP
TOTAL_STEPS = EST_STEPS_PER_EPOCH * EPOCHS

# WandB
WANDB_PROJECT = "Spedrox_llm"
USE_WANDB     = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
VOCAB_SIZE = len(tokenizer)
PAD_ID = tokenizer.pad_token_id

print(f"Vocab: {VOCAB_SIZE}, Pad ID: {PAD_ID}")
print(f"Tokens/step: {TOKENS_PER_STEP:,}")
print(f"Est steps/epoch: {EST_STEPS_PER_EPOCH:,}")
print(f"Total steps: {TOTAL_STEPS:,}")
print(f"Warmup: {WARMUP_STEPS} steps")

In [ ]:
# ======================== CLONE DATASET ========================
import oxen

if not os.path.exists(DATASET_DIR):
    print("Cloning SlimPajama-6B...")
    oxen.clone("https://hub.oxen.ai/datasets/SlimPajama-6B", DATASET_DIR)
    print("Done!")
else:
    print(f"Dataset exists at {DATASET_DIR}")

In [ ]:
# ======================== BUILD MODEL ========================
model = build_transformer(
    src_vocab_size=VOCAB_SIZE, tgt_vocab_size=VOCAB_SIZE,
    src_seq_len=MAX_SEQ_LEN, tgt_seq_len=MAX_SEQ_LEN,
    d_model=D_MODEL, N=NUM_LAYERS, h=NUM_HEADS, kv_h=KV_HEADS,
    dropout=DROPOUT, d_ff=D_FF, use_repo=USE_REPO, use_flash=USE_FLASH,
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"REPO: {'ON' if USE_REPO else 'OFF'}, Flash: {'ON' if USE_FLASH else 'OFF'}")

# Verify RMSNorm gamma is NOT zero
for name, p in model.named_parameters():
    if 'gamma' in name:
        assert p.abs().sum() > 0, f"FATAL: {name} is all zeros!"
print("[OK] RMSNorm gamma parameters are non-zero")

In [ ]:
# ======================== SANITY CHECK ========================
print("Running sanity check...")
model.train()
dummy = torch.randint(0, VOCAB_SIZE, (2, 128), device=device)

with torch.no_grad():
    x = model.tgt_embed(dummy)
    for layer in model.decoder.layers:
        x, _ = layer(x, tgt_mask=None, use_cache=False)
    x = model.decoder.norm(x)
    logits = model.project(x)

print(f"Output shape: {logits.shape}")
print(f"Output range: [{logits.min().item():.4f}, {logits.max().item():.4f}]")
print(f"Output std:   {logits.std().item():.4f}")
assert logits.std().item() > 0.01, "FATAL: Model output has near-zero variance!"
print("[OK] Sanity check passed")

del dummy, x, logits
torch.cuda.empty_cache()

In [ ]:
# ======================== DATASET & DATALOADER ========================
train_dataset = SlimPajamaDataset(
    data_dir=DATASET_DIR, tokenizer=tokenizer, max_length=MAX_SEQ_LEN,
)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    num_workers=16, pin_memory=True, prefetch_factor=2,
)
print(f"DataLoader ready (batch={BATCH_SIZE}, workers=16)")

In [ ]:
# ======================== WANDB ========================
wandb.login(key="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")

if USE_WANDB:
    wandb.init(
        project=WANDB_PROJECT,
        name=f"slimpajama_{datetime.now().strftime('%m%d_%H%M')}",
        config={
            "model": "decoder_only_transformer",
            "params": total_params,
            "d_model": D_MODEL, "layers": NUM_LAYERS,
            "heads": NUM_HEADS, "kv_heads": KV_HEADS,
            "d_ff": D_FF, "seq_len": MAX_SEQ_LEN,
            "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
            "effective_batch": BATCH_SIZE * GRAD_ACCUM,
            "lr": LEARNING_RATE, "min_lr": MIN_LR,
            "warmup": WARMUP_STEPS, "total_steps": TOTAL_STEPS,
            "dataset": "SlimPajama-6B",
            "features": ["GQA", "REPO-Attention", "Flash-Attention", "RMSNorm"],
        },
        tags=["pre-training", "slimpajama", "a100"]
    )
    wandb.watch(model, log="all", log_freq=200)
    print(f"WandB: {wandb.run.url}")

In [ ]:
# ======================== LR SCHEDULE ========================
def get_lr(step):
    """Cosine schedule with linear warmup."""
    if step < WARMUP_STEPS:
        # Linear warmup
        return LEARNING_RATE * (step + 1) / WARMUP_STEPS
    # Cosine decay
    progress = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    progress = min(progress, 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR + (LEARNING_RATE - MIN_LR) * cosine

# Quick preview
for s in [0, 10, 50, 100, 200, 500, 1000, 5000, 10000]:
    print(f"  Step {s:>6}: LR = {get_lr(s):.8f}")

In [ ]:
# ======================== TRAINING LOOP ========================

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95),
)

# BF16 on A100 (no GradScaler needed for BF16)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
scaler = GradScaler(enabled=(not use_bf16))  # Only needed for FP16

print(f"Mixed precision: {'BF16' if use_bf16 else 'FP16'}")
print(f"GradScaler: {'OFF (BF16)' if use_bf16 else 'ON (FP16)'}")

model.train()
global_step = 0
best_loss = float('inf')
last_ckpt_time = time.time()

# Set initial LR (warmup starts from near-zero)
for pg in optimizer.param_groups:
    pg['lr'] = get_lr(0)

print(f"\nStarting training...")
print(f"  Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, Accum: {GRAD_ACCUM}")
print(f"  Effective batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Peak LR: {LEARNING_RATE}, Warmup: {WARMUP_STEPS} steps")
print(f"  Est total steps: {TOTAL_STEPS:,}")
print()

for epoch in range(EPOCHS):
    total_loss = 0.0
    batch_count = 0
    micro_count = 0
    epoch_start = time.time()

    optimizer.zero_grad(set_to_none=True)

    for i, batch in enumerate(train_loader):
        # Auto-checkpoint every 2 hours
        if time.time() - last_ckpt_time >= 7200:
            avg = total_loss / max(batch_count, 1)
            save_checkpoint(model, optimizer, epoch, global_step, avg, best_loss,
                            CHECKPOINT_DIR, f"auto_epoch{epoch+1}_step{global_step}.pt")
            last_ckpt_time = time.time()

        input_ids = batch["input_ids"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        # Forward
        with autocast(device_type="cuda", dtype=amp_dtype):
            x = model.tgt_embed(input_ids)
            for layer in model.decoder.layers:
                x, _ = layer(x, tgt_mask=None, use_cache=False)
            x = model.decoder.norm(x)
            logits = model.project(x)

            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = nn.CrossEntropyLoss(ignore_index=-100)(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )
            loss = loss / GRAD_ACCUM

        # Skip bad batches
        if not torch.isfinite(loss):
            print(f"[WARN] Non-finite loss at batch {i}, skipping")
            optimizer.zero_grad(set_to_none=True)
            micro_count = 0
            continue

        # Backward
        if scaler.is_enabled():
            scaler.scale(loss).backward()
        else:
            loss.backward()

        real_loss = loss.item() * GRAD_ACCUM
        total_loss += real_loss
        batch_count += 1
        micro_count += 1

        # Optimizer step every GRAD_ACCUM micro-batches
        if micro_count >= GRAD_ACCUM:
            if scaler.is_enabled():
                scaler.unscale_(optimizer)

            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)

            if scaler.is_enabled():
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)
            global_step += 1
            micro_count = 0

            # Update LR
            lr = get_lr(global_step)
            for pg in optimizer.param_groups:
                pg['lr'] = lr

            # Log to WandB
            if USE_WANDB:
                log_dict = {
                    "train/loss": real_loss,
                    "train/lr": lr,
                    "train/grad_norm": float(grad_norm),
                    "train/step": global_step,
                    "train/epoch": epoch + 1,
                }
                if torch.cuda.is_available():
                    log_dict["system/gpu_gb"] = torch.cuda.memory_allocated() / 1e9
                wandb.log(log_dict, step=global_step)

        # Print progress
        if i % 20 == 0:
            elapsed = time.time() - epoch_start
            lr_now = optimizer.param_groups[0]['lr']
            gpu_gb = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
            print(f"E{epoch+1} B{i:>5} | loss={real_loss:.4f} | lr={lr_now:.6f} | "
                  f"step={global_step} | {elapsed:.0f}s | {gpu_gb:.1f}GB")

        if i % 50 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Flush leftover micro-batches
    if micro_count > 0:
        if scaler.is_enabled():
            scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        if scaler.is_enabled():
            scaler.step(optimizer)
            scaler.update()
        else:
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        global_step += 1

    # End of epoch
    avg_loss = total_loss / max(batch_count, 1)
    duration = time.time() - epoch_start
    print(f"\nEpoch {epoch+1}/{EPOCHS} done | avg_loss={avg_loss:.4f} | {duration:.0f}s")

    if USE_WANDB:
        wandb.log({"epoch/avg_loss": avg_loss, "epoch/duration": duration}, step=global_step)

    if avg_loss < best_loss:
        best_loss = avg_loss
        save_checkpoint(model, optimizer, epoch, global_step, avg_loss, best_loss,
                        CHECKPOINT_DIR, "best_model.pt")
        if USE_WANDB:
            wandb.log({"train/best_loss": best_loss}, step=global_step)

    save_checkpoint(model, optimizer, epoch, global_step, avg_loss, best_loss,
                    CHECKPOINT_DIR, f"epoch_{epoch+1}.pt")

# Final
print("\nTraining complete!")
save_checkpoint(model, optimizer, EPOCHS-1, global_step, avg_loss, best_loss,
                CHECKPOINT_DIR, "final_model.pt")
if USE_WANDB:
    wandb.finish()

In [ ]:
# ======================== RESUME FROM CHECKPOINT ========================
# Uncomment to resume:
"""
ckpt = torch.load(os.path.join(CHECKPOINT_DIR, "best_model.pt"), map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
optimizer.load_state_dict(ckpt['optimizer_state_dict'])
global_step = ckpt['global_step']
best_loss = ckpt['best_loss']
print(f"Resumed from step {global_step}")
"""
print("Resume cell ready (uncomment to use).")